<a href="https://colab.research.google.com/github/braysonimbambi455/CSC4110-G03/blob/main/notebooks/G03_U4_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


# Task 4: Classification
**Owner:** Agnes Watti (COM/0074/23)
**Module:** 5

In [ ]:
# Import necessary libraries for classification
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, f1_score)

# Load the cleaned dataset from Task 2.1
BASE = '/content/drive/MyDrive/CSC4110_G03'
df = pd.read_csv(f'{BASE}/data/stage1_crops.csv')

print("Data loaded successfully! Shape:", df.shape)
df.head(3)

In [ ]:
# Create classification target
y_clf = df['Item'].copy()

# Base features WITHOUT yield (weather + inputs + location + time)
base_features = ['rain_mm', 'pesticides_t', 'avg_temp', 'Year']
X_base = df[base_features].copy()

# Frequency-encode Area (country share of rows)
X_base['Area_freq'] = df['Area'].map(df['Area'].value_counts(normalize=True))

# Show class counts and percentages
print("Class counts:\n", y_clf.value_counts())
print("\nClass percentages:\n", (y_clf.value_counts(normalize=True) * 100).round(2))

In [ ]:
# Split by time: train <= 2009, test 2010-2013
train_mask = df['Year'] <= 2009
test_mask  = df['Year'] >= 2010

X_train_base, X_test_base = X_base[train_mask], X_base[test_mask]
y_train, y_test = y_clf[train_mask], y_clf[test_mask]

print("Train size:", len(X_train_base), "| Test size:", len(X_test_base))
print("\nTrain class shares (%):")
print((y_train.value_counts(normalize=True) * 100).round(2).head())
print("\nTest class shares (%):")
print((y_test.value_counts(normalize=True) * 100).round(2).head())

In [ ]:
# Scale features (fit on train only!)
scaler = StandardScaler().fit(X_train_base)
X_train_s = scaler.transform(X_train_base)
X_test_s  = scaler.transform(X_test_base)

# Fit Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_s, y_train)

# Predict and evaluate
y_pred = log_reg.predict(X_test_s)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=log_reg.classes_, yticklabels=log_reg.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()

# Save to the outputs folder (Crucial for the Graded Evidence!)
plt.savefig(f'{BASE}/outputs/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Dictionary to store results
results_clf = {}

# 1. K-Nearest Neighbors (k=3, 5, 7)
for k in [3, 5, 7]:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
    pred = knn.predict(X_test_s)
    results_clf[f'KNN_k{k}'] = {
        'accuracy': accuracy_score(y_test, pred),
        'macro_f1': f1_score(y_test, pred, average='macro')
    }

# 2. Gaussian Naive Bayes
nb = GaussianNB().fit(X_train_s, y_train)
pred_nb = nb.predict(X_test_s)
results_clf['GaussianNB'] = {
    'accuracy': accuracy_score(y_test, pred_nb),
    'macro_f1': f1_score(y_test, pred_nb, average='macro')
}

# 3. Support Vector Machine (on 5000 rows for speed)
svm = SVC(probability=True, random_state=42).fit(X_train_s[:5000], y_train[:5000])
pred_svm = svm.predict(X_test_s)
results_clf['SVM'] = {
    'accuracy': accuracy_score(y_test, pred_svm),
    'macro_f1': f1_score(y_test, pred_svm, average='macro')
}

# Add Logistic Regression to the list for comparison
results_clf['LogisticRegression'] = {
    'accuracy': accuracy_score(y_test, y_pred),
    'macro_f1': f1_score(y_test, y_pred, average='macro')
}

print(pd.DataFrame(results_clf).T.round(4))

In [ ]:
# Build the final comparison table
results_clf = pd.DataFrame(results_clf).T.sort_values('macro_f1', ascending=False)
results_clf

### Class Shares & Split
The train set (1990–2009) and test set (2010–2013) have similar class proportions. For example, Maize and Potatoes dominate the dataset, making it imbalanced. This is important because a time split can drift; if a crop appears in one set but not the other, macro F1 would be misleading.

### Comparison of Classifiers
**Logistic Regression vs KNN:** Logistic regression learns a linear decision boundary between crops in feature space; it generalises well when crops separate roughly linearly in weather/input space. KNN is instance-based — it labels a new record by the majority class of its k nearest neighbours. KNN can capture non-linear boundaries but is sensitive to feature scaling and struggles when crops overlap in feature space.

**Naive Bayes vs SVM:** Naive Bayes assumes features are independent given the class and estimates per-class Gaussians — it's fast but the independence assumption is often wrong for weather data (rain and temperature correlate). SVM finds the maximum-margin boundary; with an RBF kernel it can separate non-linear clusters but needs tuning (C, gamma) and is slower on 28k rows.

### Written Answer
**Recommended classifier:** Based on the table, [INSERT YOUR TOP MODEL NAME HERE, e.g., Random Forest or Logistic Regression] performed best.

**Why accuracy alone is misleading:** Macro F1 is the right metric here because the 10 crop classes are imbalanced — some crops (e.g., Maize, Wheat) dominate the dataset while others are rare. Accuracy alone would be misleading: a model that always predicts the majority crop can score high accuracy but near-zero F1 on rare crops. Macro F1 averages per-class F1 equally, so it penalises poor performance on minority crops.